In [ ]:
# 1. Install required packages (100% Offline Mode)
import os
import sys
import subprocess
import glob

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

offline_pkg_dir = "/kaggle/input/datasets/mduy2911/offline-packages"
print(f"Installing offline packages from: {offline_pkg_dir}...")
wheels = glob.glob(os.path.join(offline_pkg_dir, "*.whl"))

if wheels:
    cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps"] + wheels
    subprocess.run(cmd, check=True)
    print("Offline installation completed successfully!")
else:
    raise FileNotFoundError(f"No offline wheels found in {offline_pkg_dir}!")

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
# 2. TRAINING HYPERPARAMETERS (Optimized for RTX Pro 6000 Ada / 96GB VRAM)
import os

MODEL_ID = "/kaggle/input/datasets/mduy2911/model-cache"

# --- RTX Pro 6000 Hardware Optimizations ---
USE_QLORA = False
GRADIENT_CHECKPOINTING = True

# --- Training Hyperparameters ---
MAX_LENGTH = 4096            # Maximum sequence length (optimized to cover Physics prompt and response)
BATCH_SIZE = 8            # Physical batch size optimized for RTX 6000
GRADIENT_ACCUMULATION = 4  # Gradient accumulation steps (Effective batch size = 32)
EPOCHS = 2                  # Number of training epochs
LEARNING_RATE = 1e-4        # Learning rate
OUTPUT_DIR = "/kaggle/working/results"  # Output directory on Kaggle

os.environ["WANDB_MODE"] = "disabled"
USE_WANDB = False


In [ ]:
# 3. Load NL -> FOL translation datasets, Physics dataset, and Physics Prompt
import os
import json

merged_path = "/kaggle/input/datasets/mduy2911/folc-train/logic_merged_valid_augmented.json"
physics_path = "/kaggle/input/datasets/mduy2911/folc-train/physics_distillation.json"

def load_translation_dataset(path):
    samples = []
    seen_premises = set()
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    for item in data:
        nl_list = item.get("premises-NL", [])
        fol_list = item.get("premises-FOL", [])
        if not nl_list or not fol_list or len(nl_list) != len(fol_list):
            continue
        nl_serialized = "\n".join(nl_list)
        if nl_serialized in seen_premises:
            continue
        seen_premises.add(nl_serialized)
        
        sample_dict = {
            "premises-NL": nl_list, 
            "premises-FOL": fol_list,
            "example_id": item.get("example_id", ""),
            "dataset_source": item.get("dataset_source", ""),
            "question": item.get("question", ""),
            "answer": item.get("answer", "")
        }
        if "split" in item:
            sample_dict["split"] = item["split"]
        samples.append(sample_dict)
        
    print(f"Loaded {len(samples)} unique translation samples from {os.path.basename(path)}")
    return samples
def load_physics_dataset(path):
    samples = []
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    for item in data:
        inp = item.get("input", "")
        out = item.get("output", "")
        if inp and out:
            samples.append({"input": inp, "output": out})
    print(f"Loaded {len(samples)} physics samples from {os.path.basename(path)}")
    return samples

raw_samples = load_translation_dataset(merged_path)
physics_samples = load_physics_dataset(physics_path)

physics_system_prompt = """You are a precise physics reasoning agent.

TASK:
Convert a physics problem and any provided reasoning policies into executable SymPy code and a valid JSON response.

<REASONING_POLICY_OVERRIDE>

A <reasoning_policies> block may be provided.

If present:

1. Treat it as the primary reasoning guidance.

2. Follow its definitions, representations, topology rules,
   coordinate conventions, state models, and solution procedures.

3. Apply the underlying reasoning pattern rather than copying
   example expressions verbatim.

4. Override a policy only when required for:
   - physical validity
   - mathematical validity
   - SymPy executability

</REASONING_POLICY_OVERRIDE>

<OPERATING_CONSTRAINTS>

Return ONLY:

{
  "thought": "...",
  "physics_analysis": [...],
  "algebraic_reasoning": [...],
  "python_code": "...",
  "json_terminated": true
}

thought format:

<detected structure>. <activated reasoning pattern>. <solution strategy>.

thought:
- concise
- high-level
- no calculations
- no intermediate values
- no final values

physics_analysis:
- concise policy-grounded physical interpretation
- record relevant physical facts, assumptions, states, or constraints
- no calculations
- no final values

algebraic_reasoning:
- concise policy-grounded symbolic procedure
- describe the intended solve workflow
- no calculations
- no final values

python_code:
- begin with "import sympy as sp"
- use sp.Float(...) for numerical constants
- define variables before use
- solve symbolically before numeric evaluation
- evaluate norms, distances, and square roots using float(...) or .evalf()
- single-line string only
- separate statements using "; "
- no loops
- no functions
- no conditional branches
- no newline characters

Final code statements must define:

ans = [...]
unit = [...]

Requirements:
- ans must be a list
- unit must be a list
- len(ans) == len(unit)
- use raw SymPy-computed values
- do not manually round or format values
- no trailing semicolon after the final statement

Use SI base or SI derived units only.
Do not use engineering-prefix units.

Return the JSON object only.

</OPERATING_CONSTRAINTS>"""


In [ ]:
# 4. Initialize Tokenizer and Load Helper Functions
import os
import sys
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Check hardware bfloat16 support
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    compute_dtype = torch.bfloat16
    use_bf16 = True
    use_fp16 = False
    print("GPU supports bfloat16. Using bfloat16 compute (Optimal for Ampere/Ada/Hopper GPUs like RTX Pro 6000).")
else:
    compute_dtype = torch.float16
    use_bf16 = False
    use_fp16 = True
    print("Using float16 compute (Standard for Turing/Pascal/Volta GPUs or CPU).")

# Select the most optimal Attention implementation
attn_impl = "sdpa"
try:
    import flash_attn
    attn_impl = "flash_attention_2"
    print("FlashAttention-2 is installed. Using flash_attention_2.")
except ImportError:
    print("FlashAttention-2 not found. Using PyTorch Native SDPA (Scaled Dot Product Attention).")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, 
    trust_remote_code=True, 
    local_files_only=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def load_base_model():
    print("Loading a fresh instance of the base model...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=compute_dtype if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
        attn_implementation=attn_impl,
        local_files_only=True
    )
    model.config.use_cache = False
    return model

def clean_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("GPU and system memory cleaned.")

target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Optimized LoRA Configuration (Rank = 32, Alpha = 64) for logical capacity
peft_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)


In [ ]:
# 5. Format Dataset (Chat Template) and split Train/Val
import os
import json
import random
from datasets import Dataset

# Define prompt templates for flat JSON list output with strict count constraint
system_prompt_template = (
    "You convert natural-language premises into parser-safe first-order logic formulas.\n\n"
    "Output a STRICT valid JSON list of strings containing the first-order logic formulas in the exact order of the input premises.\n"
    "You must output EXACTLY the same number of formulas as the input premises. Do not skip any premises or merge them.\n\n"
    "ALLOWED OPERATORS:\n"
    "AND, OR, NOT, ->, <->, =, !=, >=, <=, >, <, ForAll, Exists\n\n"
    "QUANTIFIER RULES:\n"
    "Use nested quantifiers only. E.g., ForAll(x, ForAll(y, P(x,y)))\n\n"
    "Return JSON only."
)

user_prompt_template = (
    "Convert the following {num_premises} premises into canonical first-order logic.\n\n"
    "Premises:\n"
    "{premises}\n\n"
    "Return a JSON list of exactly {num_premises} strings containing the formulas, in the exact same order."
)

# --- Split FOL dataset before augmentation to prevent data leakage ---
has_presplit = all("split" in sample for sample in raw_samples)
if has_presplit:
    train_fol = [s for s in raw_samples if s.get("split") == "train"]
    val_fol = [s for s in raw_samples if s.get("split") == "val"]
    # Track statistics for print compatibility below
    train_orig_fol = [s for s in train_fol if "augmented" not in str(s.get("dataset_source", ""))]
    val_orig_fol = val_fol
    train_augmented_fol = [s for s in train_fol if "augmented" in str(s.get("dataset_source", ""))]
else:
    original_fol = []
    augmented_fol = []
    for sample in raw_samples:
        ds = str(sample.get("dataset_source", ""))
        if "augmented" in ds:
            augmented_fol.append(sample)
        else:
            original_fol.append(sample)

    # Shuffle original FOL samples deterministically
    random.Random(42).shuffle(original_fol)
    split_idx_fol = int(len(original_fol) * 0.9)
    train_orig_fol = original_fol[:split_idx_fol]
    val_orig_fol = original_fol[split_idx_fol:]

    # Map augmented samples back to train split
    train_orig_ids = set(x["example_id"] for x in train_orig_fol)

    def get_original_id(example_id):
        for suffix in ["_aug_var", "_perm_var", "_neg_var"]:
            if suffix in example_id:
                return example_id.split(suffix)[0]
        return example_id

    train_augmented_fol = []
    for sample in augmented_fol:
        base_id = get_original_id(sample["example_id"])
        if base_id in train_orig_ids:
            train_augmented_fol.append(sample)

    # Combine splits
    train_fol = train_orig_fol + train_augmented_fol
    val_fol = val_orig_fol

# --- Split Physics dataset deterministically ---
random.Random(42).shuffle(physics_samples)
split_idx_phys = int(len(physics_samples) * 0.9)
train_physics = physics_samples[:split_idx_phys]
val_physics = physics_samples[split_idx_phys:]

# --- Format training and validation samples ---
def format_samples(fol_list, physics_list):
    formatted = []
    
    # Format FOL translation samples
    for item in fol_list:
        nl_list = item["premises-NL"]
        fol_list_item = item["premises-FOL"]
        
        nl_content = ""
        for i, nl in enumerate(nl_list, start=1):
            nl_content += f"{i}. {nl}\n"
            
        user_prompt = user_prompt_template.format(num_premises=len(nl_list), premises=nl_content.strip())
        assistant_response = json.dumps(fol_list_item, indent=2)
        
        formatted.append({
            "messages": [
                {"role": "system", "content": system_prompt_template},
                {"role": "user", "content": user_prompt.strip()},
                {"role": "assistant", "content": assistant_response.strip()}
            ]
        })
        
    # Format Physics samples
    for item in physics_list:
        physics_input = item["input"]
        physics_output = item["output"]
        
        formatted.append({
            "messages": [
                {"role": "system", "content": physics_system_prompt},
                {"role": "user", "content": physics_input.strip()},
                {"role": "assistant", "content": physics_output.strip()}
            ]
        })
        
    return formatted

formatted_train = format_samples(train_fol, train_physics)
formatted_val = format_samples(val_fol, val_physics)

train_dataset = Dataset.from_list(formatted_train)
val_dataset = Dataset.from_list(formatted_val)

def apply_template(batch):
    texts = []
    for messages in batch["messages"]:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

train_dataset = train_dataset.map(apply_template, batched=True, remove_columns=["messages"])
val_dataset = val_dataset.map(apply_template, batched=True, remove_columns=["messages"])

# Shuffle the final HF datasets
train_dataset = train_dataset.shuffle(seed=42)
val_dataset = val_dataset.shuffle(seed=42)

print(f"FOL Train/Val Split: original train={len(train_orig_fol)}, original val={len(val_orig_fol)}")
print(f"FOL Augmented added to Train: {len(train_augmented_fol)}")
print(f"Physics Train/Val Split: train={len(train_physics)}, val={len(val_physics)}")
print(f"Total Train size: {len(train_dataset)}, Total Val size: {len(val_dataset)}")


In [ ]:
# 6. Configure SFTTrainer and start training
import os
import glob
import torch
import json
import random
import re
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback, DataCollatorForLanguageModeling
from typing import Dict, Union, Any, Optional, List, Tuple

class LossLoggingCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            if "loss" in logs:
                print(f"Step {state.global_step}: training_loss = {logs['loss']:.6f}")
            if "eval_loss" in logs:
                print(f"Step {state.global_step}: validation_loss = {logs['eval_loss']:.6f}")

class CustomDataCollator(DataCollatorForLanguageModeling):
    def __init__(self, response_template, tokenizer, *args, **kwargs):
        super().__init__(tokenizer=tokenizer, mlm=False, *args, **kwargs)
        self.response_template = response_template
        self.response_token_ids = self.tokenizer.encode(self.response_template, add_special_tokens=False)
        
    def __call__(self, examples):
        batch = super().__call__(examples)
        labels = batch["labels"]
        for i in range(labels.size(0)):
            input_ids = batch["input_ids"][i].tolist()
            response_idx = -1
            n_template = len(self.response_token_ids)
            for j in range(len(input_ids) - n_template + 1):
                if input_ids[j:j+n_template] == self.response_token_ids:
                    response_idx = j + n_template
                    break
            
            if response_idx != -1:
                labels[i, :response_idx] = -100
                
        return batch

def clean_json_response(response: str) -> str:
    response = response.strip()
    # 1. Strip markdown code block wrapping if present
    if response.startswith("```"):
        match = re.search(r"```(?:json)?\s*(.*?)\s*```", response, re.DOTALL)
        if match:
            response = match.group(1).strip()
            
    # 2. Extract first valid JSON array or object
    first_brace = response.find("{")
    first_bracket = response.find("[")
    
    if first_brace != -1 and (first_bracket == -1 or first_brace < first_bracket):
        # Extract outer JSON object (try fully closed first, fallback to open-ended)
        obj_match = re.search(r"(\{.*\})", response, re.DOTALL)
        if obj_match:
            response = obj_match.group(1).strip()
        else:
            obj_match_open = re.search(r"(\{.*)", response, re.DOTALL)
            if obj_match_open:
                response = obj_match_open.group(1).strip()
    elif first_bracket != -1:
        # Extract outer JSON array (try fully closed first, fallback to open-ended)
        array_match = re.search(r"(\[.*\])", response, re.DOTALL)
        if array_match:
            response = array_match.group(1).strip()
        else:
            array_match_open = re.search(r"(\[.*)", response, re.DOTALL)
            if array_match_open:
                response = array_match_open.group(1).strip()
                
    # 3. Auto-repair truncated JSON
    in_quote = False
    escape = False
    stack = []
    i = 0
    while i < len(response):
        char = response[i]
        if escape:
            escape = False
        elif char == '\\':
            escape = True
        elif char == '"':
            in_quote = not in_quote
        elif not in_quote:
            if char in ('{', '['):
                stack.append(char)
            elif char in ('}', ']'):
                if stack and ((char == '}' and stack[-1] == '{') or (char == ']' and stack[-1] == '[')):
                    stack.pop()
        i += 1
        
    if in_quote:
        response += '"'
    while stack:
        top = stack.pop()
        if top == '{':
            response += '}'
        elif top == '[':
            response += ']'
            
    return response

# --- OFFLINE FOL PARSER (HARDCODED FROM src/logic/reasoning/parser.py) ---
try:
    import z3
    from z3 import (
        And,
        BoolSort,
        Const,
        DeclareSort,
        Exists,
        ForAll,
        Function,
        IntSort,
        IntVal,
        Not,
        Or,
        RealSort,
        RealVal,
        StringVal,
        BoolRef,
        ExprRef,
    )
    import threading
    from typing import Dict, Iterable, List, Optional, Tuple
    
    _TOKEN_RE = re.compile(
        r"\s*(->|<->|AND|OR|NOT|IN|ForAll|Exists|>=|<=|!=|=|>|<|\(|\)|,|\+|-|\d+\.\d+|\d+|'[^']*'|[^\W\d][\w-]*)"
    )
    
    class Z3Symbols:
        def __init__(self, sort: ExprRef) -> None:
            self.sort = sort
            self.consts: Dict[str, ExprRef] = {}
            self.preds: Dict[Tuple[str, int], ExprRef] = {}
            self.funcs: Dict[Tuple[str, int], ExprRef] = {}
            self.numeric_symbols: set[str] = set()
    
        def get_const(self, name: str, sort: Optional[ExprRef] = None) -> ExprRef:
            if name in self.consts:
                return self.consts[name]
            use_sort = sort
            if use_sort is None:
                use_sort = RealSort() if name in self.numeric_symbols else self.sort
            const = Const(name, use_sort)
            self.consts[name] = const
            return const
    
        def get_pred(self, name: str, arity: int) -> ExprRef:
            key = (name, arity)
            if key in self.preds:
                return self.preds[key]
            if name in self.numeric_symbols:
                return self.get_func(name, arity, RealSort())
            pred = Function(name, *([self.sort] * arity), BoolSort())
            self.preds[key] = pred
            return pred
    
        def get_func(self, name: str, arity: int, sort: Optional[ExprRef] = None) -> ExprRef:
            key = (name, arity)
            if key in self.funcs:
                return self.funcs[key]
            use_sort = sort
            if use_sort is None:
                use_sort = RealSort() if name in self.numeric_symbols else self.sort
            func = Function(name, *([self.sort] * arity), use_sort)
            self.funcs[key] = func
            return func
            
    class TokenStream:
        def __init__(self, text: str) -> None:
            self.tokens = [t for t in _TOKEN_RE.findall(text) if t.strip()]
            self.index = 0
    
        def peek(self) -> Optional[str]:
            if self.index >= len(self.tokens):
                return None
            return self.tokens[self.index]
    
        def peek_offset(self, offset: int) -> Optional[str]:
            idx = self.index + offset
            if idx >= len(self.tokens):
                return None
            return self.tokens[idx]
    
        def next(self) -> Optional[str]:
            tok = self.peek()
            if tok is not None:
                self.index += 1
            return tok
    
        def expect(self, value: str) -> None:
            tok = self.next()
            if tok != value:
                raise ValueError(f"Expected '{value}', got '{tok}'")
                
    class FolParser:
        def __init__(self, symbols: Z3Symbols) -> None:
            self.symbols = symbols
            self.var_stack: List[Dict[str, ExprRef]] = []
    
        def parse(self, text: str) -> BoolRef:
            stream = TokenStream(text)
            expr = self._parse_implication(stream)
            return expr
    
        def _parse_implication(self, stream: TokenStream) -> BoolRef:
            left = self._parse_or(stream)
            tok = stream.peek()
            if tok == "->":
                stream.next()
                right = self._parse_implication(stream)
                return Or(Not(left), right)
            if tok == "<->":
                stream.next()
                right = self._parse_implication(stream)
                return And(Or(Not(left), right), Or(Not(right), left))
            return left
    
        def _parse_or(self, stream: TokenStream) -> BoolRef:
            left = self._parse_and(stream)
            while stream.peek() == "OR":
                stream.next()
                right = self._parse_and(stream)
                left = Or(left, right)
            return left
    
        def _parse_and(self, stream: TokenStream) -> BoolRef:
            left = self._parse_not(stream)
            while stream.peek() == "AND":
                stream.next()
                right = self._parse_not(stream)
                left = And(left, right)
            return left
    
        def _parse_not(self, stream: TokenStream) -> BoolRef:
            if stream.peek() == "NOT":
                stream.next()
                return Not(self._parse_not(stream))
            return self._parse_atom(stream)
    
        def _parse_atom(self, stream: TokenStream) -> BoolRef:
            tok = stream.peek()
            if tok is None:
                raise ValueError("Unexpected end of input")
            if tok == "(":
                stream.next()
                expr = self._parse_implication(stream)
                stream.expect(")")
                return expr
            if tok in ("ForAll", "Exists"):
                return self._parse_quantifier(stream)
            start_index = stream.index
            term = self._parse_term(stream)
            comp = stream.peek()
            if comp == "IN":
                stream.next()
                right = self._parse_term(stream)
                pred = self.symbols.get_pred("In", 2)
                return pred(term, right)
            if comp in ("=", "!=", ">=", "<=", ">", "<"):
                numeric = comp in (">=", "<=", ">", "<")
                if comp in ("=", "!="):
                    next_tok = stream.peek_offset(1)
                    if next_tok is not None and re.fullmatch(r"\d+(?:\.\d+)?", next_tok):
                        numeric = True
                if numeric and isinstance(term, BoolRef):
                    stream.index = start_index
                    term = self._parse_term(stream, prefer_numeric=True)
                    comp = stream.peek()
                stream.next()
                right = self._parse_term(stream, prefer_numeric=numeric)
                return self._build_comparison(comp, term, right)
            if not isinstance(term, BoolRef):
                raise ValueError("Predicate expected, got term")
            return term
    
        def _parse_quantifier(self, stream: TokenStream) -> BoolRef:
            quant = stream.next()
            stream.expect("(")
            var_name = stream.next()
            if var_name is None:
                raise ValueError("Missing quantified variable")
            stream.expect(",")
            var = Const(var_name, self.symbols.sort)
            self.var_stack.append({var_name: var})
            body = self._parse_implication(stream)
            self.var_stack.pop()
            stream.expect(")")
            if quant == "ForAll":
                return ForAll([var], body)
            return Exists([var], body)
    
        def _parse_term(self, stream: TokenStream, prefer_numeric: bool = False) -> ExprRef:
            left = self._parse_simple_term(stream, prefer_numeric=prefer_numeric)
            while stream.peek() in ("+", "-"):
                op = stream.next()
                right = self._parse_simple_term(stream, prefer_numeric=True)
                if op == "+":
                    left = left + right
                else:
                    left = left - right
            return left
    
        def _parse_simple_term(self, stream: TokenStream, prefer_numeric: bool = False) -> ExprRef:
            tok = stream.next()
            if tok is None:
                raise ValueError("Unexpected end of input")
            if tok == "(":
                expr = self._parse_term(stream, prefer_numeric=prefer_numeric)
                stream.expect(")")
                return expr
            if tok.startswith("'") and tok.endswith("'"):
                return StringVal(tok[1:-1])
    
            time_match = re.match(r"^Time(\d{1,2})(\d{2})?(AM|PM)$", tok, re.IGNORECASE)
            if time_match:
                hour = int(time_match.group(1))
                minute = int(time_match.group(2)) if time_match.group(2) else 0
                ampm = time_match.group(3).upper()
                if ampm == "PM" and hour < 12:
                    hour += 12
                elif ampm == "AM" and hour == 12:
                    hour = 0
                minutes = hour * 60 + minute
                return IntVal(minutes)
    
            duration_match = re.match(r"^Duration(\d+(?:\.\d+)?)(Hours|Minutes|Days)$", tok, re.IGNORECASE)
            if duration_match:
                value = float(duration_match.group(1))
                unit = duration_match.group(2).lower()
                if unit == "hours":
                    minutes = int(value * 60)
                elif unit == "days":
                    minutes = int(value * 24 * 60)
                else:
                    minutes = int(value)
                return IntVal(minutes)
    
            if tok.replace(".", "", 1).isdigit():
                if "." in tok:
                    return RealVal(tok)
                return IntVal(tok)
            var = self._lookup_var(tok)
            if var is not None:
                return var
            if stream.peek() == "(":
                stream.next()
                args = []
                if stream.peek() != ")":
                    while True:
                        args.append(self._parse_term(stream, prefer_numeric=prefer_numeric))
                        if stream.peek() == ",":
                            stream.next()
                            continue
                        break
                stream.expect(")")
                if prefer_numeric:
                    func = self.symbols.get_func(tok, len(args), RealSort())
                    return func(*args)
                pred = self.symbols.get_pred(tok, len(args))
                return pred(*args)
            if prefer_numeric:
                return self.symbols.get_const(tok, RealSort())
            return self.symbols.get_const(tok)
    
        def _build_comparison(self, op: str, left: ExprRef, right: ExprRef) -> BoolRef:
            if op == "=":
                return left == right
            if op == "!=":
                return left != right
            if op == ">=":
                return left >= right
            if op == "<=":
                return left <= right
            if op == ">":
                return left > right
            if op == "<":
                return left < right
            raise ValueError(f"Unsupported comparator: {op}")
    
        def _lookup_var(self, name: str) -> Optional[ExprRef]:
            for scope in reversed(self.var_stack):
                if name in scope:
                    return scope[name]
            return None

    def parse_formulas(formulas, sort_name="U"):
        numeric_symbols = set()
        for f in formulas:
            f_temp = f.replace("<->", " BICOND ").replace("->", " IMPLIES ")
            
            left_matches = re.finditer(r"\b([A-Za-z_][A-Za-z0-9_-]*)\s*(?:\([^()]*\))?\s*(?:>=|<=|>|<|\+|-)\b", f_temp)
            for m in left_matches:
                name = m.group(1)
                if name not in ("ForAll", "Exists", "AND", "OR", "NOT", "IN", "BICOND", "IMPLIES"):
                    numeric_symbols.add(name)
                    
            right_matches = re.finditer(r"\b(?:>=|<=|>|<|\+|-)\s*([A-Za-z_][A-Za-z0-9_-]*)\b", f_temp)
            for m in right_matches:
                name = m.group(1)
                if name not in ("ForAll", "Exists", "AND", "OR", "NOT", "IN", "BICOND", "IMPLIES"):
                    numeric_symbols.add(name)
                    
            eq_matches = re.finditer(r"\b([A-Za-z_][A-Za-z0-9_-]*)\s*(?:\([^()]*\))?\s*(?:=|!=)\s*(?:\d+(?:\.\d+)?|Time\d+[A-Za-z]+|Duration\d+[A-Z]+)\b", f_temp)
            for m in eq_matches:
                name = m.group(1)
                if name not in ("ForAll", "Exists", "AND", "OR", "NOT", "IN", "BICOND", "IMPLIES"):
                    numeric_symbols.add(name)
                    
            eq_rev_matches = re.finditer(r"\b(?:\d+(?:\.\d+)?|Time\d+[A-Za-z]+|Duration\d+[A-Z]+)\s*(?:=|!=)\s*([A-Za-z_][A-Za-z0-9_-]*)\b", f_temp)
            for m in eq_rev_matches:
                name = m.group(1)
                if name not in ("ForAll", "Exists", "AND", "OR", "NOT", "IN", "BICOND", "IMPLIES"):
                    numeric_symbols.add(name)
    
        symbols = Z3Symbols(sort=DeclareSort(sort_name))
        symbols.numeric_symbols = numeric_symbols
        
        parser = FolParser(symbols)
        formula_exprs = [parser.parse(f) for f in formulas]
        return symbols, formula_exprs

    _z3_lock = threading.Lock()
    
    def try_parse_fol(formula: str) -> tuple[bool, str]:
        with _z3_lock:
            try:
                parse_formulas([formula])
                return True, ""
            except Exception as exc:
                return False, str(exc)
                
    fol_parser_available = True
    print("Offline Z3 logic parser successfully initialized.")
except ImportError:
    fol_parser_available = False
    print("Warning: Z3 or logic parser not available. Using basic syntax checking (parentheses/operators).")
    
    def try_parse_fol(formula: str) -> tuple[bool, str]:
        return False, "Z3 solver library not installed."

# --- ACCURACY EVALUATION HELPERS ---
def compare_physics_answers(pred_ans, pred_unit, gt_ans, gt_unit):
    if pred_ans is None or gt_ans is None:
        return False
    if not isinstance(pred_ans, list):
        pred_ans = [pred_ans]
    if not isinstance(gt_ans, list):
        gt_ans = [gt_ans]
    if len(pred_ans) != len(gt_ans):
        return False
    for p_val, g_val in zip(pred_ans, gt_ans):
        try:
            p_num = float(p_val)
            g_num = float(g_val)
            if g_num == 0.0:
                if abs(p_num) >= 1e-5:
                    return False
            else:
                rel_err = abs(p_num - g_num) / abs(g_num)
                if rel_err > 0.02:  # 2% tolerance
                    return False
        except (ValueError, TypeError):
            if str(p_val).strip().lower() != str(g_val).strip().lower():
                return False
    return True

def basic_validate_fol(formula: str) -> bool:
    if formula.count("(") != formula.count(")"):
        return False
    for bad in [" and ", " or ", " not "]:
        if bad in formula:
            return False
    for bad_q in ["forall(", "exists("]:
        if bad_q in formula.lower() and not (bad_q.capitalize() in formula or "ForAll" in formula or "Exists" in formula):
            return False
    return True

# --- Z3 DOWNSTREAM QA EVALUATION HELPERS ---
def parse_mcq_options(text: str) -> dict[str, str]:
    options = {}
    pattern = r'(?:\s+|^)([A-D])[\.\)]\s+(.*?)(?=\s+[A-D][\.\)]\s+|$)'
    matches = re.findall(pattern, text, re.DOTALL)
    for opt_char, opt_text in matches:
        options[opt_char] = opt_text.strip()
    return options

def strip_question_framing(text: str) -> str:
    stripped = text.strip()
    patterns = [
        r"^(?:does it follow that|is it true that|can we conclude that|is it the case that"
        r"|do the premises imply that|does the passage imply that"
        r"|does it follow from the premises that)\s*",
    ]
    for pat in patterns:
        m = re.match(pat, stripped, re.IGNORECASE)
        if m:
            stripped = stripped[m.end():]
            break
    stripped = stripped.rstrip("?").strip()
    if stripped:
        stripped = stripped[0].upper() + stripped[1:]
    return stripped

def verify_with_z3(premises_fol: list[str], conclusion_fol: str, negate_conclusion: bool = True) -> dict:
    negated_conclusion_fol = f"NOT ({conclusion_fol})" if negate_conclusion else conclusion_fol
    all_formulas = premises_fol + [negated_conclusion_fol]
    
    try:
        symbols, exprs = parse_formulas(all_formulas)
    except Exception as parse_err:
        try:
            standardized_formulas = []
            for f_str in all_formulas:
                f_clean = f_str.replace("¬", "NOT ").replace("∧", " AND ").replace("∨", " OR ").replace("→", " -> ").replace("↔", " <-> ")
                open_count = f_clean.count("(")
                close_count = f_clean.count(")")
                if close_count < open_count:
                    f_clean = f_clean + ")" * (open_count - close_count)
                standardized_formulas.append(f_clean)
            symbols, exprs = parse_formulas(standardized_formulas)
        except Exception as parse_err_inner:
            return {"result": z3.unknown, "error": str(parse_err_inner)}
            
    premise_exprs = exprs[:-1]
    negated_conclusion_expr = exprs[-1]
    
    solver = z3.Solver()
    solver.set("timeout", 10000)
    
    for expr in premise_exprs:
        solver.add(expr)
    solver.add(negated_conclusion_expr)
    
    try:
        result = solver.check()
        return {"result": result}
    except Exception as z3_err:
        return {"result": z3.unknown, "error": str(z3_err)}

def check_answers_match(predicted, ground_truth):
    p_lower = str(predicted).strip().lower()
    gt_lower = str(ground_truth).strip().lower()
    
    yes_true = {"yes", "true"}
    no_false = {"no", "false"}
    unknown_uncertain = {"unknown", "uncertain"}
    
    if p_lower in yes_true and gt_lower in yes_true:
        return True
    if p_lower in no_false and gt_lower in no_false:
        return True
    if p_lower in unknown_uncertain and gt_lower in unknown_uncertain:
        return True
    return p_lower == gt_lower

def translate_sentences(model, tokenizer, sentences: list[str]) -> list[str]:
    if not sentences:
        return []
        
    nl_content = ""
    for i, nl in enumerate(sentences, start=1):
        nl_content += f"{i}. {nl}\n"
        
    user_prompt = user_prompt_template.format(num_premises=len(sentences), premises=nl_content.strip())
    messages = [
        {"role": "system", "content": system_prompt_template},
        {"role": "user", "content": user_prompt.strip()}
    ]
    
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
        
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    cleaned_response = clean_json_response(response)
    
    try:
        fol_list = json.loads(cleaned_response)
        if isinstance(fol_list, list) and len(fol_list) == len(sentences):
            return [str(x).strip() for x in fol_list]
    except Exception:
        pass
        
    # Fallback to translate one by one
    fallback_fol = []
    for s in sentences:
        fallback_fol.append(translate_single_sentence(model, tokenizer, s))
    return fallback_fol

def translate_single_sentence(model, tokenizer, s: str) -> str:
    user_prompt = user_prompt_template.format(num_premises=1, premises=f"1. {s}")
    messages = [
        {"role": "system", "content": system_prompt_template},
        {"role": "user", "content": user_prompt.strip()}
    ]
    
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
        
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    cleaned_response = clean_json_response(response)
    
    try:
        fol_list = json.loads(cleaned_response)
        if isinstance(fol_list, list) and len(fol_list) > 0:
            return str(fol_list[0]).strip()
    except Exception:
        pass
    return cleaned_response

GENERIC_WORDS = {
    'project', 'projects', 'python', 'code', 'standard', 'standards',
    'rule', 'rules', 'requirement', 'requirements', 'practice', 'practices',
    'convention', 'conventions', 'protocol', 'protocols', 'specification', 'specifications',
    'constraint', 'constraints', 'condition', 'conditions', 'regulation', 'regulations',
    'policy', 'policies', 'guideline', 'guidelines', 'recommendation', 'recommendations',
    'system', 'systems', 'application', 'applications', 'program', 'programs'
}

RESERVED_WORDS = {'ForAll', 'Exists', 'AND', 'OR', 'NOT', 'implies', 'IN'}

def split_camel_snake(s: str) -> list[str]:
    parts = s.split('_')
    words = []
    for part in parts:
        camel_parts = re.findall(r'[A-Z]?[a-z0-9]+|[A-Z]+(?=[A-Z][a-z0-9]|\b)', part)
        if camel_parts:
            words.extend(camel_parts)
        else:
            words.append(part)
    return words

def clean_repetitive_name(name: str) -> str:
    if len(name) <= 50:
        return name
    for length in range(4, 40):
        for i in range(len(name) - 2 * length):
            sub = name[i:i+length]
            j = i + length
            count = 1
            while name[j:j+length] == sub:
                count += 1
                j += length
            if count >= 2:
                repeated_part = sub * count
                name = name.replace(repeated_part, sub)
                return clean_repetitive_name(name)
    if len(name) > 50:
        name = name[:47] + "Trunc"
    return name

def unify_fol_predicates(formulas: list[str]) -> list[str]:
    if not formulas:
        return formulas
    predicate_pattern = r'\b([A-Za-z_][A-Za-z0-9_]*)\s*\('
    predicates = set()
    for f in formulas:
        matches = re.findall(predicate_pattern, f)
        for m in matches:
            if m not in RESERVED_WORDS:
                predicates.add(m)
    mapping = {}
    for p in predicates:
        words = split_camel_snake(p)
        core = [w for w in words if w.lower() not in GENERIC_WORDS]
        if not core:
            core = words
        canonical = "".join(w[0].upper() + w[1:] if len(w) > 0 else w for w in core)
        canonical = clean_repetitive_name(canonical)
        mapping[p] = canonical
    unified_formulas = []
    for f in formulas:
        new_f = f
        for name in sorted(mapping.keys(), key=len, reverse=True):
            canonical = mapping[name]
            if name != canonical:
                new_f = re.sub(rf'\b{name}\b', canonical, new_f)
        unified_formulas.append(new_f)
    return unified_formulas

def check_sample_with_z3(predicted_premises_fol, question, model, tokenizer):
    from z3 import unsat, sat
    
    # 1. Parse MCQ options
    options = parse_mcq_options(question)
    
    if len(options) >= 2:
        # MCQ flow
        opt_keys = sorted(options.keys())
        opt_texts = [options[k] for k in opt_keys]
        
        # Translate all options to FOL in one batch
        options_fol = translate_sentences(model, tokenizer, opt_texts)
        if len(options_fol) != len(opt_keys):
            # Fallback if mismatch
            options_fol = [options_fol[0] if len(options_fol) > 0 else ""] * len(opt_keys)
            
        # Unify predicates across both premises and options to prevent sort/predicate name mismatches
        all_fol = predicted_premises_fol + options_fol
        unified_fol = unify_fol_predicates(all_fol)
        unified_premises_fol = unified_fol[:len(predicted_premises_fol)]
        unified_options_fol = unified_fol[len(predicted_premises_fol):]
        
        entailed_options = []
        consistent_options = []
        
        for k, opt_fol in zip(opt_keys, unified_options_fol):
            if not opt_fol:
                continue
            # Check if entailed
            res = verify_with_z3(unified_premises_fol, opt_fol, negate_conclusion=True)
            if res.get("result") == unsat:
                entailed_options.append(k)
            elif res.get("result") == sat:
                # Check if consistent (not contradicted)
                res_neg = verify_with_z3(unified_premises_fol, opt_fol, negate_conclusion=False)
                if res_neg.get("result") != unsat:
                    consistent_options.append(k)
                    
        if len(entailed_options) == 1:
            return entailed_options[0]
        elif len(entailed_options) > 1:
            return entailed_options[0]
        elif len(consistent_options) >= 1:
            return consistent_options[0]
        else:
            return "Unknown"
            
    else:
        # Yes/No/Uncertain flow
        conclusion_text = strip_question_framing(question)
        conclusion_fol_list = translate_sentences(model, tokenizer, [conclusion_text])
        conclusion_fol = conclusion_fol_list[0] if conclusion_fol_list else ""
        
        if not conclusion_fol:
            return "Unknown"
            
        # Unify predicates across both premises and conclusion
        all_fol = predicted_premises_fol + [conclusion_fol]
        unified_fol = unify_fol_predicates(all_fol)
        unified_premises_fol = unified_fol[:-1]
        unified_conclusion_fol = unified_fol[-1]
        
        # Check entailment of conclusion
        res = verify_with_z3(unified_premises_fol, unified_conclusion_fol, negate_conclusion=True)
        if res.get("result") == unsat:
            return "Yes"
            
        # Check entailment of negation of conclusion
        res_neg = verify_with_z3(unified_premises_fol, unified_conclusion_fol, negate_conclusion=False)
        if res_neg.get("result") == unsat:
            return "No"
            
        return "Unknown"

def evaluate_fol_accuracy(model, tokenizer, val_samples, limit=50):
    print(f"Evaluating FOL Accuracy on {min(len(val_samples), limit)} samples...")
    correct_count = 0
    total_count = 0
    valid_json_count = 0
    syntax_valid_count = 0
    formula_correct = 0
    formula_total = 0
    
    z3_correct_count = 0
    z3_total_count = 0
    
    random_subset = random.Random(42).sample(val_samples, min(len(val_samples), limit))
    
    for item in random_subset:
        nl_list = item["premises-NL"]
        fol_list_gt = item["premises-FOL"]
        
        nl_content = ""
        for i, nl in enumerate(nl_list, start=1):
            nl_content += f"{i}. {nl}\n"
            
        user_prompt = user_prompt_template.format(num_premises=len(nl_list), premises=nl_content.strip())
        messages = [
            {"role": "system", "content": system_prompt_template},
            {"role": "user", "content": user_prompt.strip()}
        ]
        
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=1024, # Increased for safety
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
            
        generated_ids = outputs[0][inputs.input_ids.shape[1]:]
        response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        cleaned_response = clean_json_response(response)
        
        predicted_premises_fol = []
        try:
            parsed_response = json.loads(cleaned_response)
            valid_json_count += 1
            if isinstance(parsed_response, list):
                predicted_premises_fol = [str(x).strip() for x in parsed_response]
                all_formulas_valid = True
                for formula in parsed_response:
                    if fol_parser_available:
                        try:
                            is_valid, _ = try_parse_fol(str(formula))
                            if not is_valid:
                                all_formulas_valid = False
                                break
                        except Exception:
                            if not basic_validate_fol(str(formula)):
                                all_formulas_valid = False
                                break
                    else:
                        if not basic_validate_fol(str(formula)):
                            all_formulas_valid = False
                            break
                if all_formulas_valid:
                    syntax_valid_count += 1
                
                if isinstance(fol_list_gt, list):
                    if [str(x).strip() for x in parsed_response] == [str(x).strip() for x in fol_list_gt]:
                        correct_count += 1
                    
                    matched_formulas = 0
                    for p_f, g_f in zip(parsed_response, fol_list_gt):
                        if str(p_f).strip() == str(g_f).strip():
                            matched_formulas += 1
                    formula_correct += matched_formulas
                    formula_total += len(fol_list_gt)
        except Exception:
            pass
            
        # Downstream Z3 Question Answering Accuracy evaluation
        question = item.get("question", "")
        gt_answer = item.get("answer", "")
        if question and gt_answer and predicted_premises_fol:
            z3_total_count += 1
            try:
                pred_ans = check_sample_with_z3(predicted_premises_fol, question, model, tokenizer)
                if check_answers_match(pred_ans, gt_answer):
                    z3_correct_count += 1
                
                # Debug output for first sample
                if total_count < 2:
                    print(f"\n--- Z3 QA Evaluation Sample {total_count + 1} ---")
                    print(f"Question: {question}")
                    print(f"GT Answer: {gt_answer} | Z3 Predicted Answer: {pred_ans}")
                    print(f"Predicted FOL: {predicted_premises_fol}")
                    print("-" * 30)
            except Exception as z3_err:
                if total_count < 2:
                    print(f"Z3 Evaluation Error for Sample {total_count + 1}: {z3_err}")
                    
        total_count += 1
        
    em_acc = (correct_count / total_count) * 100 if total_count > 0 else 0
    formula_acc = (formula_correct / formula_total) * 100 if formula_total > 0 else 0
    json_rate = (valid_json_count / total_count) * 100 if total_count > 0 else 0
    syntax_rate = (syntax_valid_count / total_count) * 100 if total_count > 0 else 0
    z3_acc = (z3_correct_count / z3_total_count) * 100 if z3_total_count > 0 else 0
    
    print("\n=== FOL Evaluation Metrics ===")
    print(f"FOL Exact Match Accuracy (Sample Level): {em_acc:.2f}% ({correct_count}/{total_count})")
    print(f"FOL Formula Level Match Accuracy: {formula_acc:.2f}% ({formula_correct}/{formula_total})")
    print(f"FOL Syntax Validity Rate: {syntax_rate:.2f}% ({syntax_valid_count}/{total_count})")
    print(f"FOL JSON Validity Rate: {json_rate:.2f}% ({valid_json_count}/{total_count})")
    print(f"FOL Z3 Downstream QA Accuracy: {z3_acc:.2f}% ({z3_correct_count}/{z3_total_count})")
    return em_acc, json_rate, syntax_rate
def evaluate_physics_accuracy(model, tokenizer, val_samples, limit=50):
    print(f"Evaluating Physics Accuracy on {min(len(val_samples), limit)} samples...")
    correct_count = 0
    total_count = 0
    valid_json_count = 0
    python_syntax_count = 0
    exec_count = 0
    
    random_subset = random.Random(42).sample(val_samples, min(len(val_samples), limit))
    
    for item in random_subset:
        inp = item["input"]
        gt_output_str = item["output"]
        
        messages = [
            {"role": "system", "content": physics_system_prompt},
            {"role": "user", "content": inp.strip()}
        ]
        
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=1024, # Increased to prevent truncation
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
            
        generated_ids = outputs[0][inputs.input_ids.shape[1]:]
        response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        cleaned_response = clean_json_response(response)
        
        if total_count < 2:
            print(f"\n--- Physics Sample {total_count + 1} ---")
            print(f"Raw Output:\n{response}")
            print(f"Cleaned Output:\n{cleaned_response}")
            print("-" * 30)
            
        try:
            parsed_response = json.loads(cleaned_response)
            valid_json_count += 1
            code_str = parsed_response.get("python_code", "")
            
            try:
                gt_parsed = json.loads(gt_output_str)
                gt_code_str = gt_parsed.get("python_code", "")
            except Exception:
                gt_code_str = ""
                
            if code_str:
                try:
                    compile(code_str, "<string>", "exec")
                    python_syntax_count += 1
                except Exception:
                    pass

                if gt_code_str:
                    local_vars_pred = {}
                    local_vars_gt = {}
                    try:
                        import sympy as sp
                        exec(code_str, {"sp": sp}, local_vars_pred)
                        pred_ans = local_vars_pred.get("ans", None)
                        pred_unit = local_vars_pred.get("unit", None)
                        
                        exec(gt_code_str, {"sp": sp}, local_vars_gt)
                        gt_ans = local_vars_gt.get("ans", None)
                        gt_unit = local_vars_gt.get("unit", None)
                        
                        if pred_ans is not None and gt_ans is not None:
                            exec_count += 1
                            if compare_physics_answers(pred_ans, pred_unit, gt_ans, gt_unit):
                                correct_count += 1
                    except Exception:
                        pass
        except Exception as e:
            if total_count < 2:
                print(f"Failed to parse JSON: {e}")
        total_count += 1
        
    acc = (correct_count / total_count) * 100 if total_count > 0 else 0
    json_rate = (valid_json_count / total_count) * 100 if total_count > 0 else 0
    python_syntax_rate = (python_syntax_count / total_count) * 100 if total_count > 0 else 0
    exec_rate = (exec_count / total_count) * 100 if total_count > 0 else 0
    
    print("\n=== Physics Evaluation Metrics ===")
    print(f"Physics Accuracy: {acc:.2f}% ({correct_count}/{total_count})")
    print(f"Physics JSON Validity Rate: {json_rate:.2f}% ({valid_json_count}/{total_count})")
    print(f"Physics Python Syntax Validity Rate: {python_syntax_rate:.2f}% ({python_syntax_count}/{total_count})")
    print(f"Physics Code Execution Rate: {exec_rate:.2f}% ({exec_count}/{total_count})")
    return acc, json_rate, python_syntax_rate, exec_rate

def train_model(train_dataset, val_dataset, output_dir):
    clean_memory()
    
    # Mathematically exact, dynamic warmup steps calculation based on actual dataset size
    num_samples = len(train_dataset)
    effective_batch_size = BATCH_SIZE * GRADIENT_ACCUMULATION
    steps_per_epoch = num_samples // effective_batch_size
    if num_samples % effective_batch_size != 0:
        steps_per_epoch += 1
    total_steps = steps_per_epoch * EPOCHS
    
    # Target exactly 5% warmup steps of total training steps (HF v5.2 compliant integer steps)
    warmup_steps = max(1, int(total_steps * 0.05))
    print(f"Training on {num_samples} samples. Steps per epoch: {steps_per_epoch}. Total steps: {total_steps}. Dynamic warmup steps: {warmup_steps}")
    
    base_model = load_base_model()
    
    # Resume from previous checkpoints if adapter_config.json exists in output_dir
    adapter_config_path = os.path.join(output_dir, "adapter_config.json")
    if os.path.exists(adapter_config_path):
        print(f"Resuming PEFT adapter weights from {output_dir}...")
        model = PeftModel.from_pretrained(base_model, output_dir, is_trainable=True)
    else:
        print("Initializing a new PEFT adapter...")
        model = get_peft_model(base_model, peft_config)
        
    model.print_trainable_parameters()
    
    # Highly Optimized SFTConfig with Cosine Decay Scheduler & Warmup
    sft_config = SFTConfig(
        output_dir=output_dir,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",             # Cosine learning rate decay scheduler
        warmup_steps=warmup_steps,              # Compliant, dynamic warm-up steps to stabilize updates
        fp16=use_fp16,
        bf16=use_bf16,
        logging_steps=1,
        logging_first_step=True,
        optim="adamw_torch",
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        per_device_eval_batch_size=BATCH_SIZE,
        report_to="none",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=2,
        dataset_text_field="text",
        max_length=MAX_LENGTH,
        neftune_noise_alpha=None,
        dataloader_num_workers=2,
        dataloader_pin_memory=True
    )

    response_template = "<|im_start|>assistant\n"
    collator = CustomDataCollator(
        response_template=response_template, 
        tokenizer=tokenizer
    )
    
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        args=sft_config,
        data_collator=collator,
        callbacks=[LossLoggingCallback()]
    )

    checkpoints = glob.glob(os.path.join(output_dir, "checkpoint-*"))
    resume_path = None
    if checkpoints:
        checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
        resume_path = checkpoints[-1]
        print(f"Found checkpoints. Resuming training from: {resume_path}")
        
    trainer.train(resume_from_checkpoint=resume_path)
    
    print(f"Saving best validation adapter weights and tokenizer to {output_dir}...")
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print("Training completed successfully!")
    
    # --- POST-TRAINING ACCURACY EVALUATION ---
    try:
        print("\n=== Starting Post-Training Accuracy Evaluation ===")
        model.eval()
        evaluate_fol_accuracy(model, tokenizer, val_fol, limit=50)
        evaluate_physics_accuracy(model, tokenizer, val_physics, limit=50)
    except Exception as e:
        print(f"Error during post-training evaluation: {e}")
    
    # Clean up model & trainer from memory
    del trainer
    del model
    del base_model
    clean_memory()


In [ ]:
# 7. Start Training and Post-Training Evaluation
train_model(train_dataset, val_dataset, OUTPUT_DIR)